In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [5]:


from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [6]:
documents[61]

{'content': '# Hybrid Search\n\nVector search finds documents by semantic meaning, while keyword search\nfinds documents by exact word matches. Hybrid search combines both.\n\nEach search produces a ranked list with scores.\n\nWe combine them with a weighted sum:\n\n```text\nscore = alpha * vector_score + (1 - alpha) * keyword_score\n```\n\nWhen `alpha = 1`, it\'s pure vector search. When `alpha = 0`, it\'s pure\nkeyword search, and values in between give a mix.\n\n## Reciprocal Rank Fusion\n\nAnother approach is fusion: merge the ranked lists from each search\nmethod and compute a combined score based on rankings.\n\nReciprocal Rank Fusion (RRF) is a simple fusion method. The score\nfor each document is the sum of `1 / (k + rank + 1)` across all\nlists where it appears.\n\nHere is how it works with an example.\n\n- text search: `[A, B, C, D, E]`\n- vector search: `[C, B, F, G, A]`\n- they have 3 documents in common (A, B, C)\n\nWith `k = 1`:\n\n```text\nVector ranks:  C=0, B=1, F=2, G

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"content": 2.0, "filename": 0.5},
    # filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# print(type(search_results[0]))
search_results[0]['filename']
# search_results

'01-agentic-rag/lessons/14-agentic-loop.md'

In [29]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [30]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [9]:
from dotenv import load_dotenv
load_dotenv()

# from ingest import build_index
from rag_helper import RAGBase
from openai import OpenAI

# documents = load_faq_data()
# index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(usage.input_tokens)

It keeps calling the model in a `while True` loop.

Each iteration:
- sends the full `messages` history to the model,
- checks the response for any `function_call`,
- runs the tool and appends the tool output to `messages`,
- repeats if there was at least one function call.

It stops when a model response has no function calls. Then the code breaks out of the loop.
7135


In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [12]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

In [13]:
from dotenv import load_dotenv
load_dotenv()

# from ingest import build_index
from rag_helper import RAGBase
from openai import OpenAI

# documents = load_faq_data()
# index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(usage.input_tokens)

It keeps calling the model inside a `while True` loop. After each model response, the code checks whether any `function_call` items were returned:

- if there is at least one function call, it runs the tool, adds the result to `messages`, and loops again
- if there are no function calls in that turn, it `break`s and stops

So the stop condition is: **no function calls in the model’s response**.
2318
